# Remote work salary premium: causal inference

Remote roles look better paid at first glance. I wanted to know how much of that gap survives after comparing similar data-science jobs.

So I treat remote work as the exposure, salary as the outcome, and the visible job/market fields as the adjustment set. It is still observational data. The point is to make the comparison less naive, not to pretend this is an experiment.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
DATA_URL = "https://raw.githubusercontent.com/YuluDuan/Hypothesis-Testing-Data-Science-salary-comparison-in-different-location/main/ds_salaries.csv"
PROJECT_DIR = Path.cwd() if Path.cwd().name == "remote-work-salary-causal-inference" else Path("notebooks/remote-work-salary-causal-inference")
ASSET_DIR = PROJECT_DIR / "assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 180
plt.rcParams["font.family"] = "DejaVu Sans"


## 1. Data and analysis sample

The file has salary records, work arrangement, seniority, job title, company size, company location, employee residence, and year.

I keep the contrast clean: fully remote rows against onsite rows. Hybrid rows are dropped because `remote_ratio == 50` is neither one thing nor the other for this question.


In [ ]:
def job_family(title: str) -> str:
    title = str(title).lower()
    if "machine learning" in title or "ml" in title or "ai" in title:
        return "Machine Learning / AI"
    if "engineer" in title:
        return "Data Engineering"
    if "scientist" in title or "science" in title:
        return "Data Science"
    if "analyst" in title or "analytics" in title:
        return "Analytics"
    if "manager" in title or "lead" in title or "head" in title or "director" in title:
        return "Management"
    return "Other"

raw = pd.read_csv(DATA_URL)

df = raw.copy()
df = df[df["remote_ratio"].isin([0, 100])].copy()
df = df[df["salary_in_usd"].between(10_000, 500_000)].copy()
df["remote"] = (df["remote_ratio"] == 100).astype(int)
df["log_salary"] = np.log(df["salary_in_usd"])
df["job_family"] = df["job_title"].map(job_family)
df["employee_market"] = np.where(df["employee_residence"].eq("US"), "US", "Non-US")
df["company_market"] = np.where(df["company_location"].eq("US"), "US", "Non-US")
df["work_year"] = df["work_year"].astype(str)

FEATURES = [
    "work_year",
    "experience_level",
    "job_family",
    "employee_market",
    "company_market",
    "company_size",
]

sample_summary = {
    "raw_rows": int(len(raw)),
    "analysis_rows": int(len(df)),
    "remote_rows": int(df["remote"].sum()),
    "onsite_rows": int((1 - df["remote"]).sum()),
    "median_salary_remote": float(df.loc[df["remote"].eq(1), "salary_in_usd"].median()),
    "median_salary_onsite": float(df.loc[df["remote"].eq(0), "salary_in_usd"].median()),
}

sample_summary


## 2. Identification strategy

The estimand is the average effect of fully remote work on log salary.

The assumption is doing real work here: after year, seniority, job family, employee market, company market, and company size, remote and onsite rows are comparable enough to line up.

That can fail. The rest of the notebook checks where it is most likely to fail: overlap, balance, effective sample size, and missing confounding.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.8))
ax.axis("off")

nodes = {
    "Seniority": (0.08, 0.72),
    "Job family": (0.08, 0.52),
    "Market / location": (0.08, 0.32),
    "Company size": (0.08, 0.12),
    "Remote work": (0.48, 0.52),
    "Salary": (0.84, 0.52),
    "Unobserved quality\n/ negotiation": (0.48, 0.14),
}

for label, (x, y) in nodes.items():
    ax.text(
        x,
        y,
        label,
        ha="center",
        va="center",
        fontsize=12,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#F6F2E8", edgecolor="#4F5661", linewidth=1.2),
    )

def arrow(a, b, color="#4F5661", style="-"):
    ax.annotate(
        "",
        xy=nodes[b],
        xytext=nodes[a],
        arrowprops=dict(arrowstyle="->", lw=1.6, color=color, linestyle=style, shrinkA=18, shrinkB=18),
    )

for confounder in ["Seniority", "Job family", "Market / location", "Company size"]:
    arrow(confounder, "Remote work")
    arrow(confounder, "Salary")
arrow("Remote work", "Salary", color="#1B6CA8")
arrow("Unobserved quality\n/ negotiation", "Remote work", color="#9A3412", style="--")
arrow("Unobserved quality\n/ negotiation", "Salary", color="#9A3412", style="--")

ax.text(0.48, 0.86, "Observed adjustment set", ha="center", fontsize=13, weight="bold", color="#4F5661")
ax.text(0.48, 0.02, "Dashed path is the main residual threat", ha="center", fontsize=10, color="#9A3412")
fig.tight_layout()
fig.savefig(ASSET_DIR / "00_causal_dag.png", bbox_inches="tight")
plt.show()


## 3. Cross-fitted nuisance models

The main estimator is cross-fitted AIPW. Each row gets propensity and outcome predictions from models that did not train on that row.

I also keep the less fancy estimators in the report. Regression adjustment, IPW, propensity matching, and partial-linear DML are useful sanity checks when the dataset is this small.


In [ ]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURES)],
        remainder="drop",
    )


def make_outcome_model():
    return Pipeline(
        steps=[
            ("prep", make_preprocessor()),
            ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 13))),
        ]
    )


def make_propensity_model():
    return Pipeline(
        steps=[
            ("prep", make_preprocessor()),
            ("logit", LogisticRegression(max_iter=3000, solver="lbfgs")),
        ]
    )


def crossfit_nuisance(data, n_splits=5, seed=RANDOM_STATE):
    X = data[FEATURES]
    y = data["log_salary"].to_numpy()
    t = data["remote"].to_numpy()

    e_hat = np.zeros(len(data))
    m_hat = np.zeros(len(data))
    m0_hat = np.zeros(len(data))
    m1_hat = np.zeros(len(data))

    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for train_idx, test_idx in splitter.split(X, t):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y[train_idx]
        t_train = t[train_idx]

        prop = make_propensity_model()
        prop.fit(X_train, t_train)
        e_hat[test_idx] = prop.predict_proba(X_test)[:, 1]

        overall = make_outcome_model()
        overall.fit(X_train, y_train)
        m_hat[test_idx] = overall.predict(X_test)

        treated = make_outcome_model()
        control = make_outcome_model()
        treated.fit(X_train.iloc[t_train == 1], y_train[t_train == 1])
        control.fit(X_train.iloc[t_train == 0], y_train[t_train == 0])
        m1_hat[test_idx] = treated.predict(X_test)
        m0_hat[test_idx] = control.predict(X_test)

    return {
        "e": np.clip(e_hat, 0.02, 0.98),
        "m": m_hat,
        "m0": m0_hat,
        "m1": m1_hat,
    }


def pct_from_log(effect):
    return 100 * np.expm1(effect)


def aipw_scores(y, t, e, m0, m1):
    return (m1 - m0) + t * (y - m1) / e - (1 - t) * (y - m0) / (1 - e)


def summarize_scores(scores):
    ate = float(np.mean(scores))
    se = float(np.std(scores, ddof=1) / np.sqrt(len(scores)))
    return {
        "log_effect": ate,
        "pct_effect": pct_from_log(ate),
        "ci_low": pct_from_log(ate - 1.96 * se),
        "ci_high": pct_from_log(ate + 1.96 * se),
        "se_log": se,
    }

nuisance = crossfit_nuisance(df)
y = df["log_salary"].to_numpy()
t = df["remote"].to_numpy()
e = nuisance["e"]
m0 = nuisance["m0"]
m1 = nuisance["m1"]

scores = aipw_scores(y, t, e, m0, m1)
aipw = summarize_scores(scores)
propensity_auc = float(roc_auc_score(t, e))

# Regression adjustment / g-computation.
reg_data = df[FEATURES + ["remote", "log_salary"]].copy()
reg_features = FEATURES + ["remote"]
reg_pre = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURES), ("remote", "passthrough", ["remote"])],
    remainder="drop",
)
reg_model = Pipeline(steps=[("prep", reg_pre), ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 13)))])
reg_model.fit(reg_data[reg_features], reg_data["log_salary"])
x1 = reg_data[reg_features].copy(); x1["remote"] = 1
x0 = reg_data[reg_features].copy(); x0["remote"] = 0
regression_effect = float(np.mean(reg_model.predict(x1) - reg_model.predict(x0)))

# Hajek IPW using cross-fitted propensities.
w_t = t / e
w_c = (1 - t) / (1 - e)
ipw_effect = float(np.sum(w_t * y) / np.sum(w_t) - np.sum(w_c * y) / np.sum(w_c))

# Propensity-score nearest-neighbor ATT: one onsite match per treated row.
treated_idx = np.where(t == 1)[0]
control_idx = np.where(t == 0)[0]
nn = NearestNeighbors(n_neighbors=1)
nn.fit(e[control_idx].reshape(-1, 1))
_, nearest = nn.kneighbors(e[treated_idx].reshape(-1, 1))
matched_controls = control_idx[nearest[:, 0]]
matching_effect = float(np.mean(y[treated_idx] - y[matched_controls]))

# Partial-linear DML from cross-fitted residuals.
y_res = y - nuisance["m"]
t_res = t - e
dml_model = LinearRegression(fit_intercept=False)
dml_model.fit(t_res.reshape(-1, 1), y_res)
dml_effect = float(dml_model.coef_[0])

naive_effect = float(df.loc[df["remote"].eq(1), "log_salary"].mean() - df.loc[df["remote"].eq(0), "log_salary"].mean())

common_low = float(max(e[t == 1].min(), e[t == 0].min()))
common_high = float(min(e[t == 1].max(), e[t == 0].max()))
support_mask = (e >= common_low) & (e <= common_high)
trimmed_mask = support_mask & (e >= 0.05) & (e <= 0.95)
trimmed_aipw = summarize_scores(scores[trimmed_mask])

ess_treated = float((w_t.sum() ** 2) / np.sum(w_t ** 2))
ess_control = float((w_c.sum() ** 2) / np.sum(w_c ** 2))

estimates = pd.DataFrame(
    [
        {"Estimator": "Naive difference", "Log Effect": naive_effect, "Percent Effect": pct_from_log(naive_effect)},
        {"Estimator": "Regression adjustment", "Log Effect": regression_effect, "Percent Effect": pct_from_log(regression_effect)},
        {"Estimator": "IPW / Hajek", "Log Effect": ipw_effect, "Percent Effect": pct_from_log(ipw_effect)},
        {"Estimator": "Propensity matching", "Log Effect": matching_effect, "Percent Effect": pct_from_log(matching_effect)},
        {"Estimator": "Cross-fit AIPW", "Log Effect": aipw["log_effect"], "Percent Effect": aipw["pct_effect"]},
        {"Estimator": "Partial-linear DML", "Log Effect": dml_effect, "Percent Effect": pct_from_log(dml_effect)},
        {"Estimator": "Trimmed AIPW", "Log Effect": trimmed_aipw["log_effect"], "Percent Effect": trimmed_aipw["pct_effect"]},
    ]
)

estimates


## 4. Balance, overlap, and sensitivity

This is where the causal estimate earns or loses trust. I check whether weighting improves the observed covariates, whether the treatment groups share support, and how small an omitted factor would need to be to erase the result.


In [ ]:
def weighted_mean_var(x, weights):
    weights = np.asarray(weights, dtype=float)
    x = np.asarray(x, dtype=float)
    mean = np.sum(weights * x) / np.sum(weights)
    var = np.sum(weights * (x - mean) ** 2) / np.sum(weights)
    return mean, var


def smd_table(data, e_hat):
    prep = make_preprocessor()
    x_encoded = prep.fit_transform(data[FEATURES])
    names = prep.get_feature_names_out()
    t_local = data["remote"].to_numpy()
    weights = np.where(t_local == 1, 1 / e_hat, 1 / (1 - e_hat))
    rows = []
    for j, name in enumerate(names):
        col = x_encoded[:, j]
        raw_t = col[t_local == 1]
        raw_c = col[t_local == 0]
        pooled = np.sqrt((np.var(raw_t, ddof=1) + np.var(raw_c, ddof=1)) / 2)
        raw_smd = 0.0 if pooled == 0 else (raw_t.mean() - raw_c.mean()) / pooled

        mt, vt = weighted_mean_var(col[t_local == 1], weights[t_local == 1])
        mc, vc = weighted_mean_var(col[t_local == 0], weights[t_local == 0])
        pooled_w = np.sqrt((vt + vc) / 2)
        weighted_smd = 0.0 if pooled_w == 0 else (mt - mc) / pooled_w
        rows.append({"Feature": name.replace("cat__", ""), "Raw SMD": raw_smd, "Weighted SMD": weighted_smd})
    return pd.DataFrame(rows)

balance = smd_table(df, e)
balance["Max Abs SMD"] = balance[["Raw SMD", "Weighted SMD"]].abs().max(axis=1)
top_balance = balance.sort_values("Max Abs SMD", ascending=False).head(18).sort_values("Raw SMD")

heterogeneity = []
for level, group in df.assign(aipw_score=scores).groupby("experience_level"):
    if group["remote"].sum() >= 5 and (1 - group["remote"]).sum() >= 5:
        group_scores = group["aipw_score"].to_numpy()
        estimate = summarize_scores(group_scores)
        heterogeneity.append(
            {
                "Experience": level,
                "Rows": int(len(group)),
                "Remote": int(group["remote"].sum()),
                "Onsite": int((1 - group["remote"]).sum()),
                "Effect": estimate["pct_effect"],
                "CI Low": estimate["ci_low"],
                "CI High": estimate["ci_high"],
            }
        )
heterogeneity = pd.DataFrame(heterogeneity)

# Omitted-confounder tipping-point grid.
# Gamma is the omitted factor effect on log salary. Delta is its remote-onsite prevalence gap.
gamma_grid = np.linspace(0, 0.25, 51)
delta_grid = np.linspace(0, 0.40, 51)
sensitivity = aipw["log_effect"] - np.outer(gamma_grid, delta_grid)

balance.head(), heterogeneity


## 5. Report charts

These are the charts used in the GitHub report. I keep them in `assets/` so the README reads like a short write-up instead of a notebook dump.


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 6.2))
plot_est = estimates.copy()
colors = ["#8A8F98" if name != "Cross-fit AIPW" else "#1B6CA8" for name in plot_est["Estimator"]]
ax.barh(plot_est["Estimator"], plot_est["Percent Effect"], color=colors)
ax.axvline(0, color="#2B2B2B", lw=1)
ax.errorbar(
    aipw["pct_effect"],
    list(plot_est["Estimator"]).index("Cross-fit AIPW"),
    xerr=[[aipw["pct_effect"] - aipw["ci_low"]], [aipw["ci_high"] - aipw["pct_effect"]]],
    fmt="none",
    color="#111827",
    capsize=5,
    lw=1.8,
)
ax.set_title("Estimated remote salary effect by method")
ax.set_xlabel("Estimated salary effect (%)")
ax.set_ylabel("")
ax.text(aipw["pct_effect"], list(plot_est["Estimator"]).index("Cross-fit AIPW") + 0.35, "95% IF interval", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(ASSET_DIR / "01_effect_estimates.png", bbox_inches="tight")
plt.show()


In [ ]:
plot_df = df.assign(propensity=e, Work_Mode=np.where(df["remote"].eq(1), "Remote", "Onsite"))
fig, ax = plt.subplots(figsize=(10.5, 6))
sns.histplot(
    data=plot_df,
    x="propensity",
    hue="Work_Mode",
    bins=24,
    stat="density",
    common_norm=False,
    element="step",
    fill=True,
    alpha=0.28,
    palette={"Remote": "#1B6CA8", "Onsite": "#C46A2B"},
    ax=ax,
)
ax.axvspan(common_low, common_high, color="#94A3B8", alpha=0.12, label="Common support")
ax.axvline(common_low, color="#64748B", ls="--", lw=1)
ax.axvline(common_high, color="#64748B", ls="--", lw=1)
ax.set_title("Propensity overlap and common support")
ax.set_xlabel("Cross-fitted probability of being fully remote")
ax.set_ylabel("Density")
ax.text(0.02, ax.get_ylim()[1] * 0.88, f"AUC: {propensity_auc:.2f}\nTrimmed rows: {len(df) - int(trimmed_mask.sum())}", fontsize=10)
fig.tight_layout()
fig.savefig(ASSET_DIR / "02_propensity_overlap.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 7))
y_pos = np.arange(len(top_balance))
ax.scatter(top_balance["Raw SMD"], y_pos, label="Raw", color="#C46A2B", s=55)
ax.scatter(top_balance["Weighted SMD"], y_pos, label="IPW weighted", color="#1B6CA8", s=55)
for i, row in enumerate(top_balance.itertuples()):
    ax.plot([row._2, row._3], [i, i], color="#CBD5E1", lw=1)
ax.axvline(0, color="#111827", lw=1)
ax.axvline(0.1, color="#64748B", ls="--", lw=1)
ax.axvline(-0.1, color="#64748B", ls="--", lw=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_balance["Feature"])
ax.set_title("Covariate balance before and after weighting")
ax.set_xlabel("Standardized mean difference")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(ASSET_DIR / "03_covariate_balance.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.8))
het = heterogeneity.sort_values("Effect")
ax.errorbar(
    het["Effect"],
    het["Experience"],
    xerr=[het["Effect"] - het["CI Low"], het["CI High"] - het["Effect"]],
    fmt="o",
    color="#1B6CA8",
    ecolor="#94A3B8",
    elinewidth=2,
    capsize=4,
)
ax.axvline(0, color="#111827", lw=1)
ax.set_title("Cross-fit AIPW effect by seniority")
ax.set_xlabel("Estimated salary effect (%)")
ax.set_ylabel("")
for _, row in het.iterrows():
    ax.text(row["CI High"] + 1.0, row["Experience"], f"n={row['Rows']}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(ASSET_DIR / "04_effect_by_seniority.png", bbox_inches="tight")
plt.show()


In [ ]:
dist_df = df.assign(Work_Mode=np.where(df["remote"].eq(1), "Remote", "Onsite"))
fig, ax = plt.subplots(figsize=(10.5, 6))
sns.kdeplot(
    data=dist_df,
    x="salary_in_usd",
    hue="Work_Mode",
    common_norm=False,
    fill=True,
    alpha=0.25,
    palette={"Remote": "#1B6CA8", "Onsite": "#C46A2B"},
    ax=ax,
)
ax.set_xscale("log")
ax.set_title("Raw salary distribution by work mode")
ax.set_xlabel("Salary in USD, log scale")
ax.set_ylabel("Density")
fig.tight_layout()
fig.savefig(ASSET_DIR / "05_raw_salary_distribution.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 6))
mesh = ax.contourf(delta_grid, gamma_grid, pct_from_log(sensitivity), levels=20, cmap="RdBu_r")
zero = ax.contour(delta_grid, gamma_grid, sensitivity, levels=[0], colors="#111827", linewidths=2)
ax.clabel(zero, fmt={0: "effect erased"}, fontsize=9)
fig.colorbar(mesh, ax=ax, label="Bias-adjusted effect (%)")
ax.set_title("Sensitivity to an omitted binary confounder")
ax.set_xlabel("Remote vs onsite prevalence imbalance")
ax.set_ylabel("Confounder effect on log salary")
ax.text(0.02, 0.235, "Small products can erase a small AIPW estimate", fontsize=9, color="#111827")
fig.tight_layout()
fig.savefig(ASSET_DIR / "06_sensitivity_tipping_point.png", bbox_inches="tight")
plt.show()


## 6. Results summary

The final cell writes the headline numbers to `results_summary.json`. That keeps the README, charts, and notebook output tied to the same run.


In [ ]:
summary = {
    **sample_summary,
    "propensity_auc": propensity_auc,
    "common_support_low": common_low,
    "common_support_high": common_high,
    "rows_inside_common_support_and_trim": int(trimmed_mask.sum()),
    "rows_trimmed_for_overlap": int(len(df) - trimmed_mask.sum()),
    "ess_treated": ess_treated,
    "ess_control": ess_control,
    "naive_pct": pct_from_log(naive_effect),
    "regression_pct": pct_from_log(regression_effect),
    "ipw_pct": pct_from_log(ipw_effect),
    "matching_pct": pct_from_log(matching_effect),
    "dml_pct": pct_from_log(dml_effect),
    "doubly_robust_pct": aipw["pct_effect"],
    "doubly_robust_ci_low": aipw["ci_low"],
    "doubly_robust_ci_high": aipw["ci_high"],
    "trimmed_doubly_robust_pct": trimmed_aipw["pct_effect"],
    "trimmed_doubly_robust_ci_low": trimmed_aipw["ci_low"],
    "trimmed_doubly_robust_ci_high": trimmed_aipw["ci_high"],
    "omitted_confounder_log_effect_x_prevalence_gap_to_null": float(max(aipw["log_effect"], 0)),
}

(PROJECT_DIR / "results_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary
